# PTCG Agent — Implementation Plan v3 (Patched Notebook)

**Status: source of truth for Phase B.** Built on top of `reference_notebook.ipynb`
(unedited, per the v3 plan's "do not edit" rule) with the **5 addendum patches** applied
to Phase B. Phase A, Phase C, Report Phase, Scoring Clarification, and Process Fixes are
reproduced here **unchanged** from the reference notebook — nothing about them changes.

| Patch | Failure mode targeted | Cells touched |
|---|---|---|
| 1. TD(λ)-blended, label-smoothed target | Sparse terminal reward | B1c |
| 2. GELU (Leaky ReLU fallback in comments) | Dying units at 64→32 width | B1b |
| 3. AdamW + SGDR cosine restarts | Sharp-minimum overfitting on tiny noisy data | B1b, B1c |
| 4. Snapshot ensemble + game-level bagging + uncertainty gate + temperature calibration | High variance from small correlated data | B1c (new), B1d |
| 5. Options-framework macro-intents + decision entropy | Report needs a citable hierarchy, not an if/elif chain | new cell |

> **Note on citation #12 (BlendRL)** from the addendum: still flagged as worth an
> independent spot-check before it goes in the Strategy report — everything else here is
> textbook-tier and used only for its well-known mechanism (TD learning, AdamW, SGDR,
> bagging, snapshot ensembles, Wilson interval, temperature scaling, options framework),
> not for any borrowed empirical claim.


## Competition Structure (Two Tracks, One Agent)

| Track | Deliverable | Deadline | Weight | Status |
|-------|-------------|----------|--------|--------|
| **Simulation** | `submission.tar.gz` (`main.py` + `deck.csv` + `cg/`) | Aug 16 23:59 UTC | Skill rating via TrueSkill/Elo | ✅ Baseline submitted (heuristic-only, ~664 tier) |
| **Strategy** | 2,000-word report analyzing the agent's design, decisions, reasoning | Sept 6 entry / Sept 13 final | Judged by humans; **gatekeeper to $240K Tokyo finals** | ❌ Does not exist |

The Strategy report needs **real ablation data** from the Simulation agent. Hard
dependency chain: Phase A → B → C → Report.


## Current Agent — Honest Assessment

`main.py` is a **pure heuristic scoring engine**; the neural scaffolding is still inert in
production until these patches are validated and wired in. Two heuristic engines with
similar priority orderings converge to similar ratings — **capped near ~664 by
construction**. Breaking past that ceiling requires the patches below, which require real
training data, which requires Phase A / Task 1 (still unclosed).

### Unverified Assumptions (High Risk, unchanged)
- `attackId` as a local index — assumed, never confirmed against a real observation
- `energies` field format — assumed raw ints, never inspected from wire data
- No real observation has been captured and manually inspected despite `GameLogger` being wired


In [ ]:
# Environment setup (unchanged from reference notebook)
import json
import math
import random
import shutil
import time
from pathlib import Path

import numpy as np

USE_MOCK_ENGINE = True
try:
    import cg  # noqa: F401  (real competition package, only present on Kaggle harness)
    USE_MOCK_ENGINE = False
except ImportError:
    pass

print(f"USE_MOCK_ENGINE = {USE_MOCK_ENGINE}")


## Phase A: Close the Verification Loop (unchanged)

Still the blocker. **None of the 5 patches below matter until Task 1 is done against a
real observation.**


In [ ]:
# Phase A / Task 1: capture + manually inspect one real observation (unchanged)

def mock_observation():
    """Stand-in shape ONLY -- do not trust until Task 1 runs against a real observation."""
    return {
        "active": {"hp": 180, "maxHp": 220, "energies": [1, 1, 1, 0], "attacks": [
            {"attackId": 0, "name": "Aura Sphere", "cost": [1, 1]},
            {"attackId": 1, "name": "Power Blast", "cost": [1, 1, 1, 1]},
        ]},
        "bench": [],
        "opponent": {"active": {"hp": 130, "maxHp": 200}, "bench": [], "prizes": 4},
        "hand": [],
        "deckCount": 34,
        "prizes": 5,
    }

def capture_and_inspect(observation, log_path="game_log.jsonl"):
    record = {"ts": time.time(), "observation": observation}
    with open(log_path, "a") as f:
        f.write(json.dumps(record) + "\n")
    print("--- Field inventory (Phase A / Task 1 checklist) ---")
    print(f"attackId values (check: local index?): "
          f"{[a['attackId'] for a in observation['active']['attacks']]}")
    print(f"energies type (check: raw ints or objects?): {type(observation['active']['energies'][0]).__name__}")
    print(f"top-level keys present: {sorted(observation.keys())}")
    return record

obs = mock_observation()
_ = capture_and_inspect(obs)


In [ ]:
# Phase A / Task 2: StateEncoder (unchanged -- 84-dim, indexed, auditable)

STATE_DIM = 84
ENERGY_TYPES = ["fighting", "colorless", "psychic", "fire", "water", "lightning"]

def _safe_ratio(hp, max_hp):
    return 0.0 if not max_hp else max(0.0, min(1.0, hp / max_hp))

def build_state_vector(observation) -> np.ndarray:
    v = np.zeros(STATE_DIM, dtype=np.float32)
    i = 0
    me = observation["active"]
    opp = observation["opponent"]["active"]

    v[i] = _safe_ratio(me["hp"], me["maxHp"]); i += 1
    v[i] = _safe_ratio(opp["hp"], opp["maxHp"]); i += 1

    v[i] = min(len(observation.get("bench", [])), 5) / 5.0; i += 1
    v[i] = min(len(observation.get("hand", [])), 10) / 10.0; i += 1
    v[i] = min(observation.get("deckCount", 0), 60) / 60.0; i += 1
    v[i] = min(len(observation["opponent"].get("bench", [])), 5) / 5.0; i += 1
    v[i] = 0.0; i += 1  # opponent hand size -- hidden info placeholder
    v[i] = 0.0; i += 1  # opponent deck count -- confirm field in Task 1

    my_prizes = observation.get("prizes", 6)
    opp_prizes = observation["opponent"].get("prizes", 6)
    v[i] = my_prizes / 6.0; i += 1
    v[i] = opp_prizes / 6.0; i += 1
    v[i] = (opp_prizes - my_prizes) / 6.0; i += 1

    energies = me.get("energies", [])
    for t in range(len(ENERGY_TYPES)):
        v[i] = (energies[t] if t < len(energies) else 0) / 4.0; i += 1

    for _ in range(4):
        v[i] = 0.0; i += 1  # weakness/resistance placeholders

    for _ in range(4):
        v[i] = 0.0; i += 1  # AttackPlan threat-level placeholders

    for _ in range(5):
        v[i] = 0.0; i += 1  # bench HP ratio placeholders

    while i < STATE_DIM:
        v[i] = 0.0; i += 1

    assert i == STATE_DIM
    return v

def validate_state_vector(v: np.ndarray):
    assert v.shape == (STATE_DIM,)
    assert not np.isnan(v).any()
    assert np.all((v >= -1.0) & (v <= 1.0))
    return True

sv = build_state_vector(obs)
validate_state_vector(sv)
print(f"state vector shape={sv.shape}, non-zero dims={int(np.count_nonzero(sv))}/{STATE_DIM}")


In [ ]:
# Phase A / Task 3: heuristic quick wins (unchanged)

ITEM_WEIGHTS = {"Ultra Ball": 5700, "Nest Ball": 5650, "Switch": 5550, "Potion": 5500}

def energy_type_match_bonus(attacker_type: str, energy_type: str) -> int:
    return 150 if attacker_type == energy_type else 0

def bench_priority_bonus(bench_count: int, hand_has_basics: bool) -> int:
    if not hand_has_basics:
        return 0
    return max(0, (3 - bench_count)) * 100

print(ITEM_WEIGHTS)


## Phase B: The Only Phase That Moves Rating — **5 Patches Applied**

Same goal as before (break the ~664 heuristic ceiling); the mechanics below are revised
per the ML/DL addendum. Loss form and BCE gradient shape are unchanged in spirit — only
the **target**, **activation**, **optimizer**, and **how many models vote** change.


In [ ]:
# Phase B / B1a: game-level 80/20 split (unchanged)

def load_game_log(path="game_log.jsonl"):
    records = []
    p = Path(path)
    if not p.exists():
        return records
    with open(p) as f:
        for line in f:
            records.append(json.loads(line))
    return records

def group_train_val_split(records, val_frac=0.2, seed=0):
    game_ids = sorted({r.get("game_id", "unassigned") for r in records})
    rng = random.Random(seed)
    rng.shuffle(game_ids)
    n_val_games = max(1, int(len(game_ids) * val_frac)) if game_ids else 0
    val_ids = set(game_ids[:n_val_games])
    train = [r for r in records if r.get("game_id", "unassigned") not in val_ids]
    val = [r for r in records if r.get("game_id", "unassigned") in val_ids]
    return train, val

records = load_game_log()
train_records, val_records = group_train_val_split(records)
print(f"loaded {len(records)} records -> train={len(train_records)} val={len(val_records)}")


### Patch 2 + 3 — `ValueNet` with GELU and an external `AdamWOptimizer`

Activation swapped from ReLU to GELU (Hendrycks & Gimpel, 2016) to remove the dying-unit
risk at 64→32 width. `step()` is removed from the class itself; optimization is now
handled by a separate `AdamWOptimizer` (Loshchilov & Hutter, 2019) so the same net class
can be paired with any optimizer, and so a cosine-annealed LR schedule (SGDR — Loshchilov
& Hutter, 2016) can drive it from outside.


In [ ]:
# Phase B / B1b -- PATCHED: GELU activation, AdamW-compatible params, no built-in step()

def gelu(x):
    # tanh approximation -- standard, numerically stable, no scipy dependency needed
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

def gelu_grad(x):
    cdf = 0.5 * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))
    pdf = np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)
    return cdf + x * pdf

def leaky_relu(x, alpha=0.01):
    # Fallback: if GELU's derivative is ever judged too fiddly under time pressure,
    # swap gelu/gelu_grad calls below for leaky_relu/leaky_relu_grad -- one-line change.
    return np.where(x > 0, x, alpha * x)

def leaky_relu_grad(x, alpha=0.01):
    return np.where(x > 0, 1.0, alpha)


class ValueNet:
    def __init__(self, in_dim=84, h1=64, h2=32, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, np.sqrt(2 / in_dim), size=(in_dim, h1)).astype(np.float32)
        self.b1 = np.zeros(h1, dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2 / h1), size=(h1, h2)).astype(np.float32)
        self.b2 = np.zeros(h2, dtype=np.float32)
        self.W3 = rng.normal(0, np.sqrt(2 / h2), size=(h2, 1)).astype(np.float32)
        self.b3 = np.zeros(1, dtype=np.float32)

    def params(self):
        return {"W1": self.W1, "b1": self.b1, "W2": self.W2, "b2": self.b2,
                "W3": self.W3, "b3": self.b3}

    def n_params(self):
        return sum(p.size for p in self.params().values())

    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = gelu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = gelu(z2)
        z3 = a2 @ self.W3 + self.b3
        yhat = 1 / (1 + np.exp(-z3))
        return yhat, (X, z1, a1, z2, a2, z3, yhat)

    def backward(self, cache, y, l2=1e-4):
        X, z1, a1, z2, a2, z3, yhat = cache
        n = X.shape[0]
        y = y.reshape(-1, 1)

        dz3 = (yhat - y) / n                    # BCE + sigmoid combined gradient -- unchanged
        dW3 = a2.T @ dz3 + l2 * self.W3
        db3 = dz3.sum(axis=0)

        da2 = dz3 @ self.W3.T
        dz2 = da2 * gelu_grad(z2)                # was: da2 * (z2 > 0)
        dW2 = a1.T @ dz2 + l2 * self.W2
        db2 = dz2.sum(axis=0)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * gelu_grad(z1)                # was: da1 * (z1 > 0)
        dW1 = X.T @ dz1 + l2 * self.W1
        db1 = dz1.sum(axis=0)

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}


class AdamWOptimizer:
    """Loshchilov & Hutter, 2019 -- decoupled weight decay: the wd term is applied
    directly to the parameter, not folded into the adaptive-LR gradient step."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=1e-4):
        self.lr, self.b1, self.b2, self.eps, self.wd = lr, betas[0], betas[1], eps, wd
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, params, grads, lr=None):
        self.t += 1
        lr = self.lr if lr is None else lr
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            m_hat = self.m[k] / (1 - self.b1 ** self.t)
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            params[k] -= lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.wd * params[k])


def cosine_lr(epoch, T_max, lr_min=1e-5, lr_max=1e-3):
    """SGDR cosine schedule (Loshchilov & Hutter, 2016). T_max = epochs per restart."""
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * (epoch % T_max) / T_max))

_probe_net = ValueNet()
print(f"ValueNet parameter count: {_probe_net.n_params()} (order-of-magnitude match to ~7.5K budget)")


### Patch 1 — TD(λ)-blended, label-smoothed target

Requires per-game **trajectories** (turn-ordered states), not iid (state, outcome) pairs,
so the network's own next-state estimate has something meaningful to bootstrap from. The
synthetic smoke-test data generator below is upgraded accordingly — replace it with real
turn-ordered `game_log.jsonl` records once Phase A logging is live (each record needs a
`game_id` and a `turn` index so trajectories can be reconstructed in order).


In [ ]:
# Phase B / B1c-data -- PATCHED: turn-ordered synthetic trajectories (for TD bootstrap)

def synthesize_trajectories(n_games=40, decisions_per_game=12, seed=0):
    """Per game: a turn-ordered list of state vectors + one terminal outcome.
    Replace with real trajectories reconstructed from game_log.jsonl (grouped by
    game_id, sorted by turn) once Phase A logging is live."""
    rng = np.random.default_rng(seed)
    games = []
    for g in range(n_games):
        outcome = float(rng.integers(0, 2))
        states = [rng.normal(outcome - 0.5, 0.3, size=STATE_DIM).astype(np.float32)
                  for _ in range(decisions_per_game)]
        games.append({"game_id": g, "states": states, "outcome": outcome})
    return games

def flatten_trajectories(games):
    """Returns parallel arrays: X (state_t), X_next (state_{t+1} or zeros), terminal
    mask, outcome, game_id -- everything train() needs to build the TD-blended target."""
    X, X_next, terminal, outcome, gid = [], [], [], [], []
    for game in games:
        states = game["states"]
        for t, s in enumerate(states):
            X.append(s)
            gid.append(game["game_id"])
            outcome.append(game["outcome"])
            if t + 1 < len(states):
                X_next.append(states[t + 1])
                terminal.append(0.0)
            else:
                X_next.append(np.zeros(STATE_DIM, dtype=np.float32))
                terminal.append(1.0)
    return (np.array(X), np.array(X_next), np.array(terminal, dtype=np.float32),
            np.array(outcome, dtype=np.float32), np.array(gid))

games = synthesize_trajectories()
X, X_next, terminal, outcome, gid = flatten_trajectories(games)
val_mask = gid >= int(0.8 * gid.max())
print(f"{len(games)} games -> {len(X)} decisions, {int(terminal.sum())} terminal states")


In [ ]:
# Phase B / B1c -- PATCHED: TD(lambda)-blended + label-smoothed target,
# AdamW + SGDR training loop, snapshot saving at each restart trough.

def make_td_target(net, X_next, terminal, outcome, beta, epsilon=0.05):
    """y'' = smooth( beta*outcome + (1-beta)*V_theta(next_state), eps ).
    Terminal states always use the pure outcome (there is no next state to bootstrap
    from), matching standard TD(0)/TD(lambda) boundary handling."""
    v_next, _ = net.forward(X_next)
    v_next = v_next.ravel()
    y_blend = beta * outcome + (1 - beta) * v_next
    y_blend = np.where(terminal > 0.5, outcome, y_blend)
    y_smoothed = y_blend * (1 - epsilon) + 0.5 * epsilon
    return y_smoothed.astype(np.float32)


def validation_win_rate(net, X_val, y_val_outcome, threshold=0.5):
    yhat, _ = net.forward(X_val)
    pred = (yhat.ravel() >= threshold).astype(np.float32)
    return float((pred == y_val_outcome).mean())


def train_one_model(X_train, X_next_train, term_train, outc_train,
                     X_val, outc_val,
                     seed=0, restarts=3, epochs_per_restart=40, batch_size=32,
                     lr_min=1e-5, lr_max=1e-3, wd=1e-4, l2=1e-4, label_eps=0.05):
    """Trains one ValueNet with AdamW + SGDR, TD-blended label-smoothed targets.
    Returns the trained net plus the list of param snapshots taken at each restart
    trough (Patch 4's free snapshot ensemble)."""
    net = ValueNet(seed=seed)
    opt = AdamWOptimizer(net.params(), wd=wd)
    rng = np.random.default_rng(seed)
    n = X_train.shape[0]
    total_epochs = restarts * epochs_per_restart
    snapshots = []

    for epoch in range(total_epochs):
        beta = max(0.5, 1.0 - epoch / (2 * total_epochs))  # anneal MC -> TD, 1.0 -> 0.5
        lr = cosine_lr(epoch, epochs_per_restart, lr_min=lr_min, lr_max=lr_max)

        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            y_target = make_td_target(net, X_next_train[idx], term_train[idx],
                                       outc_train[idx], beta, epsilon=label_eps)
            yhat, cache = net.forward(X_train[idx])
            grads = net.backward(cache, y_target, l2=l2)
            opt.step(net.params(), grads, lr=lr)

        if epoch % epochs_per_restart == epochs_per_restart - 1:
            snapshots.append({k: v.copy() for k, v in net.params().items()})

    val_wr = validation_win_rate(net, X_val, outc_val)
    return net, snapshots, val_wr


net, snapshots, val_wr = train_one_model(
    X[~val_mask], X_next[~val_mask], terminal[~val_mask], outcome[~val_mask],
    X[val_mask], outcome[val_mask],
)
print(f"single-model validation win-rate: {val_wr:.3f}, "
      f"{len(snapshots)} SGDR snapshots captured for free")


### Patch 4 — Game-level bagging + snapshot ensemble + uncertainty-gated tie-break + calibration

`k=5` bootstrap resamples **at the game level** (same independence unit as the train/val
split -- resampling at the decision level would leak exactly the way a decision-level
split does), each trained with the SGDR loop above, contributing its 3 restart snapshots
to the ensemble. Total ensemble size: `5 * 3 = 15` tiny models (~450KB total, still
trivial). Temperature scaling (Guo et al., 2017) is fit afterward on the held-out
validation split.


In [ ]:
# Phase B / new cell -- PATCHED: game-level bagging + full ensemble + calibration

def bootstrap_game_resample(games, k=5, seed=0):
    """k bootstrap resamples at the GAME level, not the decision level."""
    rng = random.Random(seed)
    game_ids = [g["game_id"] for g in games]
    by_id = {g["game_id"]: g for g in games}
    resamples = []
    for i in range(k):
        sampled_ids = rng.choices(game_ids, k=len(game_ids))
        resamples.append([by_id[gid] for gid in sampled_ids])
    return resamples


def train_ensemble(games, val_frac=0.2, k=5, seed=0):
    """Returns a flat list of (net) ensemble members: k bagged models x however
    many SGDR snapshots each contributes."""
    game_ids = sorted({g["game_id"] for g in games})
    rng = random.Random(seed)
    shuffled = game_ids[:]
    rng.shuffle(shuffled)
    n_val = max(1, int(len(shuffled) * val_frac))
    val_ids = set(shuffled[:n_val])
    train_games = [g for g in games if g["game_id"] not in val_ids]
    val_games = [g for g in games if g["game_id"] in val_ids]

    Xv, Xv_next, termv, outv, _ = flatten_trajectories(val_games)

    resamples = bootstrap_game_resample(train_games, k=k, seed=seed)
    ensemble_nets = []
    val_wrs = []
    for i, resample in enumerate(resamples):
        Xt, Xt_next, termt, outt, _ = flatten_trajectories(resample)
        model, model_snapshots, val_wr = train_one_model(
            Xt, Xt_next, termt, outt, Xv, outv, seed=seed + i,
        )
        ensemble_nets.append(model)   # final weights
        val_wrs.append(val_wr)
        # Snapshot members are also usable: rehydrate a ValueNet per snapshot if a
        # larger/more diverse ensemble is wanted -- omitted here to keep inference
        # cost predictable (one forward pass per bagged model, not per snapshot).

    return ensemble_nets, val_wrs, (Xv, outv)


ensemble_nets, member_val_wrs, (Xv, outv) = train_ensemble(games, k=5, seed=0)
print(f"trained {len(ensemble_nets)} bagged models, "
      f"per-member validation win-rate: {[f'{w:.3f}' for w in member_val_wrs]}")


def ensemble_predict(nets, X):
    preds = np.array([net.forward(X)[0].ravel() for net in nets])  # (k, n)
    return preds.mean(axis=0), preds.std(axis=0)


def fit_temperature(nets, X_val, y_val, T_grid=None):
    """Guo et al., 2017 -- fit a single scalar T on held-out data to calibrate the
    ensemble-mean probability. Grid search since we have no autograd."""
    if T_grid is None:
        T_grid = np.linspace(0.5, 3.0, 26)
    mean_pred, _ = ensemble_predict(nets, X_val)
    z = np.log(np.clip(mean_pred, 1e-6, 1 - 1e-6) / np.clip(1 - mean_pred, 1e-6, 1 - 1e-6))
    best_T, best_nll = 1.0, float("inf")
    for T in T_grid:
        p = 1 / (1 + np.exp(-z / T))
        p = np.clip(p, 1e-6, 1 - 1e-6)
        nll = -np.mean(y_val * np.log(p) + (1 - y_val) * np.log(1 - p))
        if nll < best_nll:
            best_nll, best_T = nll, T
    return best_T

temperature = fit_temperature(ensemble_nets, Xv, outv)
print(f"calibrated temperature T={temperature:.2f} (fit on validation split)")


### Wiring — uncertainty-gated ensemble tie-breaker

Same ±500 heuristic margin as before, but the override now additionally requires the
ensemble to **agree** (`std(ŷ) ≤ τ`). If the ensemble disagrees, defer to the symbolic
engine — the neuro-symbolic contract stays honest: the network only overrides when it's
both close *and* confident.


In [ ]:
# Phase B / B1d -- PATCHED: ensemble + uncertainty gate replaces the single-net tie-break

TIE_MARGIN = 500
UNCERTAINTY_TAU = 0.1

def resolve_with_ensemble(scored_options, nets, state_of_option_fn, temperature=1.0,
                           tie_margin=TIE_MARGIN, tau=UNCERTAINTY_TAU):
    """scored_options: list of (heuristic_score, option). Overrides the heuristic
    top pick only if (a) contenders are within tie_margin AND (b) the ensemble's
    std on the winning option's win-probability is <= tau. Otherwise defers to the
    heuristic engine's own top choice."""
    if not scored_options:
        return None
    scored_options = sorted(scored_options, key=lambda t: t[0], reverse=True)
    top_score, top_option = scored_options[0]
    contenders = [opt for score, opt in scored_options if top_score - score <= tie_margin]

    if len(contenders) == 1:
        return contenders[0]

    states = np.stack([state_of_option_fn(opt) for opt in contenders])
    mean_pred, std_pred = ensemble_predict(nets, states)
    # apply temperature to the mean logit for a calibrated probability
    z = np.log(np.clip(mean_pred, 1e-6, 1 - 1e-6) / np.clip(1 - mean_pred, 1e-6, 1 - 1e-6))
    calibrated = 1 / (1 + np.exp(-z / temperature))

    best_idx = int(np.argmax(calibrated))
    if std_pred[best_idx] > tau:
        return top_option  # ensemble disagrees -- defer to the symbolic engine
    return contenders[best_idx]

dummy_options = [(9000, "evolve_active"), (8950, "evolve_bench"), (6500, "play_supporter")]
chosen = resolve_with_ensemble(dummy_options, ensemble_nets,
                                lambda opt: build_state_vector(obs), temperature=temperature)
print("ensemble tie-breaker chose:", chosen)


### Patch 5 — Named macro-intents (Options framework) + decision-entropy telemetry

Zero compute cost: relabels the existing `score_option()` priority buckets into named
intents with initiation conditions (Sutton, Precup & Singh, 1999), and adds a per-turn
entropy signal so the Strategy report's explainability section has a real computed
number, not an anecdote.


In [ ]:
# Phase B / new cell -- PATCHED: macro-intents + decision entropy

MACRO_INTENTS = {
    "LETHAL":    {"initiation": "KO wins game",                          "scores": (50000, 50000)},
    "AGGRO_KO":  {"initiation": "KO is feasible",                        "scores": (10000, 13000)},
    "ATTACK":    {"initiation": "no better option, attack available",    "scores": (8500, 9999)},
    "DEVELOP":   {"initiation": "bench < 3 or evolution available",      "scores": (6000, 8499)},
    "RESOURCE":  {"initiation": "supporter/item in hand",                "scores": (5500, 6499)},
    "POWER_UP":  {"initiation": "energy deficit on attacker",            "scores": (4000, 5499)},
    "STABILIZE": {"initiation": "HP < 40% and healthy bench ready",      "scores": (4200, 4200)},
    "PASS":      {"initiation": "nothing useful",                        "scores": (0, 999)},
}

def macro_intent_for_score(score: int) -> str:
    for name, spec in MACRO_INTENTS.items():
        lo, hi = spec["scores"]
        if lo <= score <= hi:
            return name
    return "UNKNOWN"

def decision_entropy(scores, temperature=1000.0):
    """H = -sum(p_i log p_i) over softmax-normalized scores. Cheap per-turn
    'how contested was this decision' signal for the Strategy report."""
    s = np.array(scores, dtype=np.float64)
    s = s - s.max()
    exp_s = np.exp(s / temperature)
    p = exp_s / exp_s.sum()
    return float(-np.sum(p * np.log(p + 1e-12)))

example_scores = [9000, 8950, 6500, 100]
print("macro intents:", [macro_intent_for_score(s) for s in example_scores])
print(f"decision entropy: {decision_entropy(example_scores):.3f}")


### B2. One-ply lookahead (unchanged from reference notebook)


In [ ]:
# Phase B / B2: one-ply lookahead scaffold (unchanged)

def one_ply_lookahead(state, legal_options, evaluate_terminal_fn):
    if USE_MOCK_ENGINE:
        return max(legal_options, key=lambda opt: evaluate_terminal_fn(state, opt))
    raise NotImplementedError("wire to real cg.search_begin/search_step on the Kaggle harness")

def dummy_evaluate(state, opt):
    return hash((str(state)[:8], opt)) % 100

print(one_ply_lookahead("state", ["attack_a", "attack_b"], dummy_evaluate))


## Phase C: Stabilize, Don't Innovate (unchanged)

Non-regression gate is unaffected by the patches -- it gates on *match outcomes*, not on
which model produced the agent's decisions.


In [ ]:
# Phase C: Wilson score interval + non-regression gate (unchanged)

def wilson_interval(wins: int, n: int, z: float = 1.96):
    if n == 0:
        return 0.0, 0.0, 0.0
    p_hat = wins / n
    denom = 1 + z**2 / n
    center = p_hat + z**2 / (2 * n)
    margin = z * math.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    low = (center - margin) / denom
    high = (center + margin) / denom
    return p_hat, low, high

def non_regression_gate(wins: int, n: int, threshold: float = 0.55, min_n: int = 30):
    if n < min_n:
        return False, f"only {n} matches played (need >= {min_n})"
    p_hat, low, high = wilson_interval(wins, n)
    verdict = low >= threshold
    msg = (f"win-rate={p_hat:.3f} (95% CI [{low:.3f}, {high:.3f}]), "
           f"threshold={threshold} -> {'PROMOTE' if verdict else 'HOLD'}")
    return verdict, msg

promote, msg = non_regression_gate(wins=18, n=30)
print("raw 60% @ n=30:", msg)


## Report Phase (structure unchanged; ablation now covers the ensemble)

Ablation claim (4) now has three variants to report, not two: heuristic-only vs.
single-net tie-breaker vs. ensemble+TD-target tie-breaker -- so the report can show
whether the addendum patches actually moved the needle over the simpler Phase B baseline,
not just over the pure heuristic.


In [ ]:
# Report Phase: ablation harness -- now with 3 variants

def run_ablation(agent_variants: dict, n_matches_per_variant: int = 30):
    results = {}
    for name, play_match_fn in agent_variants.items():
        wins = sum(1 for _ in range(n_matches_per_variant) if play_match_fn() == 1)
        p_hat, low, high = wilson_interval(wins, n_matches_per_variant)
        results[name] = {"win_rate": p_hat, "ci_low": low, "ci_high": high, "n": n_matches_per_variant}
    return results

def _mock_match_heuristic_only():
    return random.random() < 0.50
def _mock_match_single_net_tiebreaker():
    return random.random() < 0.58
def _mock_match_ensemble_tiebreaker():
    return random.random() < 0.62  # placeholder -- replace with real match results

ablation_results = run_ablation({
    "heuristic_only": _mock_match_heuristic_only,
    "single_net_tiebreaker": _mock_match_single_net_tiebreaker,
    "ensemble_td_tiebreaker": _mock_match_ensemble_tiebreaker,
})
for name, r in ablation_results.items():
    print(f"{name:26s} win_rate={r['win_rate']:.3f} CI=[{r['ci_low']:.3f}, {r['ci_high']:.3f}] n={r['n']}")


## Scoring System Clarification (unchanged)

| System | What It Is | Units | Visible To |
|--------|-----------|-------|------------|
| `score_option()` internal | Arbitrary integer ranking of legal options within one decision | Points | Only our agent |
| Competition skill rating | TrueSkill/Elo Bayesian rating | Rating (starts at 600) | Kaggle leaderboard |

Sample agent sits at **664.2**; daily median climbed ~628 → ~1,180 over 38 days.


## Process Fixes (unchanged)

1. Backup `main.py` before every teamwork run
2. Incremental outputs, not one final dump
3. Keep the current working submission as a separate, untouched copy


In [ ]:
# Process fix: pre-run backup helper (unchanged)

def backup_before_teamwork_run(main_py_path="main.py", backup_dir="backups"):
    src = Path(main_py_path)
    backup_root = Path(backup_dir)
    backup_root.mkdir(exist_ok=True)
    if not src.exists():
        print(f"WARNING: {main_py_path} not found -- nothing to back up")
        return None
    stamp = time.strftime("%Y%m%d_%H%M%S")
    dst = backup_root / f"main_{stamp}.py"
    shutil.copy2(src, dst)
    print(f"backed up {src} -> {dst}")
    return dst

print("backup helper defined")


## Immediate Next Step (unchanged)

**Phase A, Task 1 is still the blocker.** None of the 5 patches above matter until a real
observation is captured and inspected. Run `capture_and_inspect` against a real
observation (not `mock_observation()`) before trusting any number this notebook prints.
